In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [33]:
PROJECT_ROOT = Path("../").resolve()

MASTER_DATASET = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deploypilot_ai_ci_cd_pipeline_dataset.csv"
)

print(PROJECT_ROOT)
print(MASTER_DATASET)

E:\SE(ICBT)\Top Up\Final Project\deploypilot-ai
E:\SE(ICBT)\Top Up\Final Project\deploypilot-ai\data\processed\deploypilot_ai_ci_cd_pipeline_dataset.csv


In [4]:
master_df = pd.read_csv(MASTER_DATASET)

master_df.head()

,pipeline_id,run_id,timestamp,ci_tool,repository,branch,commit_hash,author,language,os,...,retry_count,is_flaky_test,rollback_triggered,incident_created,result,commit_size,files_changed,warnings,tests_failed,previous_failure_rate
0,ado-feature-store-nightly-regression,20260619.24,2026-04-28T22:44:00,Azure DevOps,mlops/feature-store,develop,b61ba350873af319212adfadb5dc5a6dfd856d83,akash_mlops,Python,windows-latest,...,0,False,False,False,FAIL,3,7,2,0,0.207
1,jenkins-e2e-test-suite-release,e2e-test-suite/feature-pagos#22855,2026-04-17T06:51:00,Jenkins,qa/e2e-test-suite,feature/pagos,99568baae90d2f0893079353e02d580f9f454ef2,data_pipeline_bot,C++,ubuntu-latest,...,3,False,False,False,FAIL,62,22,24,0,0.309
2,gha-analytics-etl-build-test-deploy,118001076740-1,2026-04-25T16:49:00,GitHub Actions,data/analytics-etl,release/2026.07,2d023683742031c29712d7960be4e9ddc2d1533f,data_pipeline_bot,Go,windows-latest,...,0,False,False,False,FAIL,11,5,0,0,0.093
3,circleci-billing-api-security-scan,9e8ff9ec-6682-5f20-8fa4-9cead21c4a5e,2025-12-19T06:58:00,CircleCI,finance/billing-api,bugfix/payment-webhook,abc701170de64d4c390f57460c1e991999612d0f,junior_juan,Java,ubuntu-latest,...,0,False,False,False,PASS,28,11,1,0,0.046
4,jenkins-deploypilot-ai-deploy-staging,deploypilot-ai/bugfix-flaky-login-test#30551,2026-05-12T01:32:00,Jenkins,devops/deploypilot-ai,bugfix/flaky-login-test,bb4190851815f62664fa796aa525e364b001004b,nimal_backend,C++,macos-latest,...,3,False,False,True,FAIL,35,11,3,0,0.821


In [5]:
print("Shape:")
print(master_df.shape)


Shape:
(20000, 31)


In [6]:
print(master_df.columns.tolist())

['pipeline_id', 'run_id', 'timestamp', 'ci_tool', 'repository', 'branch', 'commit_hash', 'author', 'language', 'os', 'cloud_provider', 'build_duration_sec', 'test_duration_sec', 'deploy_duration_sec', 'failure_stage', 'failure_type', 'error_code', 'error_message', 'severity', 'cpu_usage_pct', 'memory_usage_mb', 'retry_count', 'is_flaky_test', 'rollback_triggered', 'incident_created', 'result', 'commit_size', 'files_changed', 'warnings', 'tests_failed', 'previous_failure_rate']


In [7]:
master_df.isnull().sum()

pipeline_id                  0
run_id                       0
timestamp                    0
ci_tool                      0
repository                   0
branch                       0
commit_hash                  0
author                       0
language                     0
os                           0
cloud_provider             732
build_duration_sec           0
test_duration_sec            0
deploy_duration_sec          0
failure_stage            10000
failure_type             10000
error_code               10000
error_message                0
severity                 10000
cpu_usage_pct                0
memory_usage_mb              0
retry_count                  0
is_flaky_test                0
rollback_triggered           0
incident_created             0
result                       0
commit_size                  0
files_changed                0
warnings                     0
tests_failed                 0
previous_failure_rate        0
dtype: int64

In [8]:
master_df["result"].value_counts()


result
FAIL    10000
PASS    10000
Name: count, dtype: int64

In [9]:
fail_df = master_df[
    master_df["result"] == "FAIL"
].copy()

print(fail_df.shape)

(10000, 31)


In [10]:
fail_df["failure_type"].value_counts()

failure_type
Configuration Error      1000
Security Scan Failure    1000
Dependency Error         1000
Network Error            1000
Permission Error         1000
Resource Exhaustion      1000
Test Failure             1000
Deployment Failure       1000
Build Failure            1000
Timeout                  1000
Name: count, dtype: int64

In [11]:
model_1_df = master_df.copy()

print(model_1_df.shape)

(20000, 31)


In [12]:
model_2_df = fail_df[
    fail_df["failure_type"].notna()
    &
    fail_df["error_message"].notna()
].copy()

print(model_2_df.shape)

(10000, 31)


In [13]:
MODEL_2_DATASET = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "model_2_failure_type_training_dataset.csv"
)

model_2_df.to_csv(
    MODEL_2_DATASET,
    index=False
)

print("Saved:")
print(MODEL_2_DATASET)

Saved:
E:\SE(ICBT)\Top Up\Final Project\deploypilot-ai\data\processed\model_2_failure_type_training_dataset.csv


In [14]:
model_1_features = [
    "commit_size",
    "files_changed",
    "warnings",
    "tests_failed",
    "previous_failure_rate",
    "build_duration_sec",
    "test_duration_sec",
    "deploy_duration_sec",
    "cpu_usage_pct",
    "memory_usage_mb",
    "retry_count",
    "ci_tool",
    "branch",
    "language",
    "os",
    "cloud_provider"
]

missing_features = [
    col
    for col in model_1_features
    if col not in master_df.columns
]

print("Missing Features:")
print(missing_features)

Missing Features:
[]


In [15]:
print(
    "error_message" in model_2_df.columns
)

print(
    "failure_type" in model_2_df.columns
)

True
True


In [16]:
summary_text = f"""
DEPLOYPILOT AI
PHASE 1 DATASET SUMMARY

Master Dataset Shape:
{master_df.shape}

PASS/FAIL Distribution:
{master_df['result'].value_counts()}

FAILURE TYPE DISTRIBUTION:
{fail_df['failure_type'].value_counts()}

MODEL 1 DATASET SHAPE:
{model_1_df.shape}

MODEL 2 DATASET SHAPE:
{model_2_df.shape}
"""

summary_path = (
    PROJECT_ROOT
    / "reports"
    / "metrics"
    / "phase_1_dataset_summary.txt"
)

with open(summary_path, "w") as f:
    f.write(summary_text)

print(summary_path)

E:\SE(ICBT)\Top Up\Final Project\deploypilot-ai\reports\metrics\phase_1_dataset_summary.txt


In [34]:
import sys

sys.path.append(
    r"e:\SE(ICBT)\Top Up\Final Project\deploypilot-ai"
)

In [35]:
from api.log_preprocessor import clean_log

raw_log = """
2026-07-01 10:44:21
/home/runner/project/test_login.py
AssertionError expected 200 got 500
"""

print("RAW:")
print(raw_log)

print("\nCLEANED:")
print(clean_log(raw_log))

RAW:

2026-07-01 10:44:21
/home/runner/project/test_login.py
AssertionError expected 200 got 500


CLEANED:
assertionerror expected got


In [36]:
raw_log = """
ModuleNotFoundError:
No module named pandas

Build ID 98234567
"""

print(clean_log(raw_log))

modulenotfounderror no module named pandas build id
